# Task_Classification_Procedure

This notebook will do only the LLM task-classification pipeline:

1.   Load the frozen provisional classification file.
2.   Exclude the pilot tasks from the validation sampling frame.
3.   Stratify the remaining tasks by provisional classification.
4.   Draw a fixed validation sample using a reproducible random seed, with sufficient representation of rare categories.
5.   Record sampling probabilities and weights.
6.   Save and lock the master validation sample.
7.   Calibrate the two human coders using the excluded pilot tasks.
8.   Create a blind coder file containing only task information and the frozen rubric.
9.   Have both coders classify the validation tasks independently.
10.  Calculate coder–coder agreement before adjudication:

     percentage agreement

     Cohen’s κ

     confusion matrix

11.  Adjudicate coder disagreements and create the human reference labels.
12.  Apply the prespecified systematic-rubric-error rule.
13.  Compare the AI provisional labels with the adjudicated human labels.
14.  Calculate weighted population-level validation estimates where appropriate.
15.  Save and freeze the validation-results dataset.
16.  Use the frozen Pass-1 classifications for the analytical dataset unless the systematic-error rule requires a full rerun.

Step 1 — Load the locked task universe

In [ ]:
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm
import json
from google.colab import userdata
from openai import OpenAI
from sklearn.metrics import cohen_kappa_score


ROOT = Path("/content/drive/MyDrive/GenAI-task-affordance-labor-outcomes-dev/02_Task_Data_and_Classification")

TASK_UNIVERSE_FILE = (
    ROOT / "Output" / "Occupational_Task_Universe_ONET30_v2.csv"
)

task_universe = pd.read_csv(TASK_UNIVERSE_FILE)

#display(task_universe.head())

Step 2 — confirm the locked universe and unique task IDs.

In [ ]:
print(f"Tasks: {len(task_universe):,}")
print(f"Unique Task IDs: {task_universe['task_id'].nunique():,}")
print(f"Occupations: {task_universe['soc6'].nunique()}")

Tasks: 13,508
Unique Task IDs: 13,508
Occupations: 604


Step 3 — select the 20-task pilot

In [ ]:
SEED = 42

pilot_tasks = (
    task_universe
    .sample(n=20, random_state=SEED)
    .reset_index(drop=True)
)

Step 4 — Save the pilot permanently

In [ ]:
PILOT_FILE = Path(
    ROOT / "Output" /
    "Occupational_Task_Classification_Pilot_v1.csv"
)

if PILOT_FILE.exists():
    pilot_tasks = pd.read_csv(PILOT_FILE)
    print("Locked pilot already exists — existing 20 tasks loaded.")

else:
    pilot_tasks = (
        task_universe
        .sample(n=20, random_state=42)
        .reset_index(drop=True)
    )

    pilot_tasks.to_csv(PILOT_FILE, index=False)
    print(f"Pilot created and saved: {PILOT_FILE}")

Pilot created and saved: /content/drive/MyDrive/GenAI-task-affordance-labor-outcomes-dev/02_Task_Data_and_Classification/Output/Occupational_Task_Classification_Pilot_v1.csv


Step 5: finalize the classification rubric and prompt.

| Label                           | Decision rule                                                                                                                                                       |
| ------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Limited GenAI applicability** | GenAI cannot materially perform the core task or provide meaningful assistance to the central work outcome.                                                         |
| **Augmentation-oriented**       | GenAI can materially assist with the task, but substantive human judgment, action, interaction, or expertise remains necessary to produce the central work outcome. |
| **Automation-oriented**         | GenAI can perform most of the core task, with the human role mainly limited to initiating, monitoring, approving, or handling exceptions.                           |
| **Indeterminate mechanism**     | The task statement does not provide enough information to distinguish the mechanism reliably, or multiple mechanisms are equally central.                           |


In [ ]:
RUBRIC_VERSION = "v6"

SYSTEM_PROMPT = """
You are classifying an occupational task according to the dominant
Generative AI task-affordance mechanism.

Assess the task using generally available Generative AI capabilities that
operate on digital information and content. Do not assume robotics, physical
actuation, or specialized non-GenAI automation.

Physical execution is outside the GenAI capability scope. If the central work
activity is primarily physical, GenAI support through information retrieval,
preparation, drafting, documentation, or other surrounding digital activities
does not by itself constitute augmentation. Classify such a task as Limited
GenAI applicability unless GenAI can directly perform a substantive
non-physical component explicitly contained in the task as stated.

The classification is based on two principles:
- Automation: GenAI performs or takes over most of the substantive task activity.
- Augmentation: GenAI and the human jointly perform substantive components of
  the task.

Classify the TASK AS STATED, focusing on its central work activity rather than
activities that merely surround or support it.

Use exactly one label:

1. Limited GenAI applicability
   GenAI cannot directly perform a substantive component of the central work
   activity. Assistance limited to peripheral preparation, information
   retrieval, drafting, scheduling, documentation, or general support does not
   qualify as augmentation when those activities are not themselves part of
   the central task.

2. Augmentation-oriented
   GenAI can directly perform a substantive component of the central work
   activity, but meaningful human contribution remains necessary to complete
   the task. The remaining human contribution may involve judgment, expertise,
   interpersonal interaction, accountability, decision-making, or execution of
   other substantive task components.

3. Automation-oriented
   GenAI can perform most of the substantive central work activity such that
   the central task outcome can largely be produced by GenAI. Human involvement
   is primarily initiation, monitoring, verification, approval, or exception
   handling.

4. Indeterminate mechanism
   The task statement does not provide sufficient information to determine the
   dominant mechanism reliably, or two or more classifications remain similarly
   plausible after applying the decision rules.

Decision rules:
- Evaluate the central work activity explicitly described in the task statement.
- GenAI must directly perform part of the stated task for the task to qualify
  as augmentation or automation.
- For Augmentation-oriented or Automation-oriented classification, GenAI must
  be capable of directly executing a substantive part of the central action
  expressed by the task statement, not merely improving, informing, preparing
  for, or supporting the human who performs that action.
- For tasks whose central action is inherently relational, interpersonal,
  institutional, authoritative, or physical, digital analysis or content
  generation surrounding that action does not by itself satisfy the
  direct-performance requirement.
- Useful assistance somewhere in the surrounding workflow is not sufficient.
- If GenAI only supports activities peripheral to the central task ->
  Limited GenAI applicability.
- If GenAI directly performs a substantive part of the central task while the
  human remains materially involved -> Augmentation-oriented.
- If GenAI can perform most of the substantive central task ->
  Automation-oriented.
- If the dominant mechanism cannot be determined reliably, or two or more
  classifications remain similarly plausible after applying the decision
  rules -> Indeterminate mechanism.

Preparation, information retrieval, drafting, scheduling, and documentation
are considered peripheral only when they support a different central work
activity. If one of these activities is itself the central activity explicitly
stated in the task, direct GenAI performance of that activity counts toward
augmentation or automation.

Do not infer additional task components that are not stated or reasonably
implied by the occupational task statement.

Ambiguity rule:
- Set ambiguity_flag = True only when the task statement is materially
  underspecified or when two or more classifications remain similarly plausible
  after applying the decision rules.
- "Materially underspecified" means the task statement omits information needed
  to apply the decision rules and choose a label. For example, it may name an
  activity without indicating whether GenAI or the human performs the
  substantive component. It does not mean that a task is merely brief.
- Indeterminate mechanism must always have ambiguity_flag = True.
- Otherwise set ambiguity_flag = False.

Return:
- one label,
- a concise rationale explaining how GenAI relates to the central work activity,
- an ambiguity flag.
"""

In [ ]:
def build_task_prompt(row):
    return f"""
Occupation: {row['occupation_title']}
Task: {row['task_statement']}
"""

Step 6 — Configure the model

In [ ]:
api_key = userdata.get("OPENAI_API_KEY")

if not api_key:
    raise RuntimeError(
        "OPENAI_API_KEY not found. Add it to Colab Secrets and enable notebook access."
    )

client = OpenAI(api_key=api_key)

MODEL = "gpt-5.6-luna"
REASONING_EFFORT = "low"

In [ ]:
OUTPUT_SCHEMA = {
    "type": "json_schema",
    "name": "task_classification",
    "schema": {
        "type": "object",
        "properties": {
            "label": {
                "type": "string",
                "enum": [
                    "Limited GenAI applicability",
                    "Augmentation-oriented",
                    "Automation-oriented",
                    "Indeterminate mechanism"
                ]
            },
            "rationale": {
                "type": "string"
            },
            "ambiguity_flag": {
                "type": "boolean"
            }
        },
        "required": ["label", "rationale", "ambiguity_flag"],
        "additionalProperties": False
    },
    "strict": True
}

In [ ]:
row = pilot_tasks.iloc[0]

response = client.responses.create(
    model=MODEL,
    reasoning={"effort": REASONING_EFFORT},
    text={"format": OUTPUT_SCHEMA},
    input=[
        {
            "role": "developer",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": build_task_prompt(row)
        }
    ]
)

result = json.loads(response.output_text)

print(result)

{'label': 'Limited GenAI applicability', 'rationale': 'The central task is physically fabricating dental restorations or impressions from physical preliminary impressions. Generative AI may assist with digital guidance or documentation, but it cannot directly perform the required physical fabrication under the stated task.', 'ambiguity_flag': False}


In [ ]:
pilot_outputs = []

for i, (_, row) in enumerate(pilot_tasks.iterrows(), start=1):

    response = client.responses.create(
        model=MODEL,
        reasoning={"effort": REASONING_EFFORT},
        text={"format": OUTPUT_SCHEMA},
        input=[
            {
                "role": "developer",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": build_task_prompt(row)
            }
        ]
    )

    result = json.loads(response.output_text)

    pilot_outputs.append({
        "task_id": row["task_id"],
        "label": result["label"],
        "rationale": result["rationale"],
        "ambiguity_flag": result["ambiguity_flag"]
    })

    print(f"{i:02d}/20  {result['label']}")

01/20  Limited GenAI applicability
02/20  Limited GenAI applicability
03/20  Augmentation-oriented
04/20  Augmentation-oriented
05/20  Limited GenAI applicability
06/20  Augmentation-oriented
07/20  Augmentation-oriented
08/20  Limited GenAI applicability
09/20  Limited GenAI applicability
10/20  Augmentation-oriented
11/20  Augmentation-oriented
12/20  Limited GenAI applicability
13/20  Limited GenAI applicability
14/20  Limited GenAI applicability
15/20  Limited GenAI applicability
16/20  Limited GenAI applicability
17/20  Limited GenAI applicability
18/20  Limited GenAI applicability
19/20  Augmentation-oriented
20/20  Augmentation-oriented


In [ ]:
pilot_results = pilot_tasks.merge(
    pd.DataFrame(pilot_outputs),
    on="task_id",
    how="left"
)

,task_id,occupation_title,task_statement,label,rationale,ambiguity_flag
0,562,Dental Assistants,Fabricate temporary restorations or custom imp...,Limited GenAI applicability,The central activity is physically fabricating...,False
1,6686,"Career/Technical Education Teachers, Secondary...","Confer with parents or guardians, other teache...",Limited GenAI applicability,The central activity is conferring and resolvi...,False
2,14680,"Computer Occupations, All Other","Develop or approve project plans, schedules, o...",Augmentation-oriented,"GenAI can substantively draft project plans, s...",False
3,3759,Social and Human Service Assistants,Interview individuals or family members to com...,Augmentation-oriented,GenAI can directly support substantive parts o...,False
4,5675,"Business Teachers, Postsecondary","Participate in student recruitment, registrati...",Limited GenAI applicability,The central activities involve participating i...,False
5,98,Training and Development Specialists,"Monitor, evaluate, or record training activiti...",Augmentation-oriented,"GenAI can directly analyze training records, s...",False
6,7210,Natural Sciences Managers,Determine scientific or technical goals within...,Augmentation-oriented,GenAI can substantively assist in formulating ...,False
7,11564,Elevator and Escalator Installers and Repairers,"Assemble, install, repair, and maintain elevat...",Limited GenAI applicability,The central activities are physically assembli...,False
8,3507,Actuaries,Testify in court as expert witness or to provi...,Limited GenAI applicability,The central activity is testifying in court as...,False
9,18770,"Library Assistants, Clerical","Perform clerical activities, such as answering...",Augmentation-oriented,GenAI can directly perform substantive digital...,False


In [ ]:
PILOT_RESULTS_FILE = (
   ROOT / "Output" /
    "Occupational_Task_Classification_Pilot_v6.csv"
)

if PILOT_RESULTS_FILE.exists():
    print("Locked v6 pilot results already exist — file not overwritten.")
else:
    pilot_results.to_csv(PILOT_RESULTS_FILE, index=False)
    print(f"Saved and locked: {PILOT_RESULTS_FILE}")

Saved and locked: /content/drive/MyDrive/GenAI-task-affordance-labor-outcomes-dev/02_Task_Data_and_Classification/Output/Occupational_Task_Classification_Pilot_v6.csv


After the classification rubric was finalized and frozen, pilot-task classifications were descriptively compared with published O*NET task-level annotations from Eloundou et al. (2024). This comparison was used as an external plausibility check only and did not inform subsequent modification of the frozen rubric or replace the study’s independent human validation procedure.

Step 7 — Pass 1

In [ ]:
PASS1_CHECKPOINT = Path(
   ROOT / "Output" /
    "Occupational_Task_Classification_Pass1_v6.jsonl"
)
PASS2_CHECKPOINT = Path(
    ROOT / "Output" /
    "Occupational_Task_Classification_Pass2_v6.jsonl"
)

In [ ]:
completed = set()

if PASS1_CHECKPOINT.exists():
    with open(PASS1_CHECKPOINT, "r") as f:
        for line in f:
            if line.strip():
                completed.add(json.loads(line)["task_id"])

print(f"Already completed: {len(completed):,}")
print(f"Remaining: {len(task_universe) - len(completed):,}")

if len(completed) == len(task_universe):
    print("Pass 1 is complete and LOCKED — no classification run executed.")

else:
    for _, row in tqdm(
        task_universe.iterrows(),
        total=len(task_universe),
        desc="Pass 1"
    ):
        task_id = int(row["task_id"])

        if task_id in completed:
            continue

        response = client.responses.create(
            model=MODEL,
            reasoning={"effort": REASONING_EFFORT},
            text={"format": OUTPUT_SCHEMA},
            input=[
                {"role": "developer", "content": SYSTEM_PROMPT},
                {"role": "user", "content": build_task_prompt(row)}
            ]
        )

        result = json.loads(response.output_text)

        output = {
            "task_id": task_id,
            "pass1_label": result["label"],
            "pass1_rationale": result["rationale"],
            "pass1_ambiguity_flag": result["ambiguity_flag"],
            "pass1_response_id": response.id
        }

        with open(PASS1_CHECKPOINT, "a") as f:
            f.write(json.dumps(output) + "\n")

        completed.add(task_id)

Already completed: 10,163
Remaining: 3,345


Pass 1:   0%|          | 0/13508 [00:00<?, ?it/s]

In [ ]:
with open(PASS1_CHECKPOINT, "r") as f:
    saved = sum(1 for line in f if line.strip())

print(f"Saved tasks: {saved:,}")

Saved tasks: 13,508


In [ ]:
pass1 = pd.read_json(PASS1_CHECKPOINT, lines=True)

print("Rows:", len(pass1))
print("Unique task IDs:", pass1["task_id"].nunique())
print("Duplicate task IDs:", pass1["task_id"].duplicated().sum())

Rows: 13508
Unique task IDs: 13508
Duplicate task IDs: 0


In [ ]:
completed = set()

if PASS2_CHECKPOINT.exists():
    with open(PASS2_CHECKPOINT, "r") as f:
        for line in f:
            if line.strip():
                completed.add(json.loads(line)["task_id"])

print(f"Already completed: {len(completed):,}")
print(f"Remaining: {len(task_universe) - len(completed):,}")

if len(completed) == len(task_universe):
    print("Pass 2 is complete and LOCKED — no classification run executed.")

else:
    for _, row in tqdm(
        task_universe.iterrows(),
        total=len(task_universe),
        desc="Pass 2"
    ):
        task_id = int(row["task_id"])

        if task_id in completed:
            continue

        response = client.responses.create(
            model=MODEL,
            reasoning={"effort": REASONING_EFFORT},
            text={"format": OUTPUT_SCHEMA},
            input=[
                {"role": "developer", "content": SYSTEM_PROMPT},
                {"role": "user", "content": build_task_prompt(row)}
            ]
        )

        result = json.loads(response.output_text)

        output = {
            "task_id": task_id,
            "pass2_label": result["label"],
            "pass2_rationale": result["rationale"],
            "pass2_ambiguity_flag": result["ambiguity_flag"],
            "pass2_response_id": response.id
        }

        with open(PASS2_CHECKPOINT, "a") as f:
            f.write(json.dumps(output) + "\n")

        completed.add(task_id)

Already completed: 0
Remaining: 1,125


Pass 2:   0%|          | 0/1125 [00:00<?, ?it/s]

In [ ]:
with open(PASS2_CHECKPOINT, "r") as f:
    saved = sum(1 for line in f if line.strip())

print(f"Saved tasks: {saved:,}")

Saved tasks: 13,508


Step 9 — compare Pass 1 vs. Pass 2

In [ ]:
pass1 = pd.read_json(PASS1_CHECKPOINT, lines=True)
pass2 = pd.read_json(PASS2_CHECKPOINT, lines=True)

comparison = pass1.merge(
    pass2,
    on="task_id",
    how="inner"
)

comparison["agreement_flag"] = (
    comparison["pass1_label"] == comparison["pass2_label"]
)

print(f"Matched tasks: {len(comparison):,}")
print(f"Agreement: {comparison['agreement_flag'].sum():,}")
print(f"Disagreement: {(~comparison['agreement_flag']).sum():,}")
print(f"Agreement rate: {comparison['agreement_flag'].mean():.1%}")

Matched tasks: 13,508
Agreement: 12,298
Disagreement: 1,210
Agreement rate: 91.0%


In [ ]:
display(
    pd.crosstab(
        comparison["pass1_label"],
        comparison["pass2_label"],
        margins=True
    )
)

pass2_label,Augmentation-oriented,Automation-oriented,Indeterminate mechanism,Limited GenAI applicability,All
pass1_label,,,,,
Augmentation-oriented,6099,172,18,363,6652
Automation-oriented,195,494,0,7,696
Indeterminate mechanism,14,1,17,7,39
Limited GenAI applicability,419,6,8,5688,6121
All,6727,673,43,6065,13508


In [ ]:
kappa = cohen_kappa_score(
    comparison["pass1_label"],
    comparison["pass2_label"]
)

print(f"Repeated-run Cohen's kappa: {kappa:.3f}")

Repeated-run Cohen's kappa: 0.837


Step 10

Pass 1 = provisional label

Pass 2 = reproducibility check

In [ ]:
# Step 10 — Create provisional classification file from Pass 1 only

pass1 = pd.read_json(PASS1_CHECKPOINT, lines=True)

provisional = task_universe.merge(
    pass1[["task_id", "pass1_label"]],
    on="task_id",
    how="left"
)

provisional = provisional.rename(
    columns={"pass1_label": "provisional_label"}
)

# Classification metadata
provisional["rubric_version"] = RUBRIC_VERSION
provisional["model"] = MODEL
provisional["reasoning_effort"] = REASONING_EFFORT

# Integrity check
print(f"Tasks: {len(provisional):,}")
print(f"Unique task IDs: {provisional['task_id'].nunique():,}")
print(f"Missing labels: {provisional['provisional_label'].isna().sum()}")

display(
    provisional[
        [
            "task_id",
            "occupation_title",
            "task_statement",
            "provisional_label"
        ]
    ].head()
)

Tasks: 13,508
Unique task IDs: 13,508
Missing labels: 0


,task_id,occupation_title,task_statement,provisional_label
0,8823,Chief Executives,Direct or coordinate an organization's financi...,Limited GenAI applicability
1,8824,Chief Executives,"Confer with board members, organization offici...",Limited GenAI applicability
2,8827,Chief Executives,"Prepare budgets for approval, including those ...",Augmentation-oriented
3,8826,Chief Executives,"Direct, plan, or implement policies, objective...",Limited GenAI applicability
4,8834,Chief Executives,Prepare or present reports concerning activiti...,Augmentation-oriented


In [ ]:
PROVISIONAL_FILE = Path(
    ROOT / "Output" /
    "Occupational_Task_Classification_Provisional_v6.csv"
)

if PROVISIONAL_FILE.exists():
    print("Provisional v6 file already exists — file not overwritten.")
else:
    provisional.to_csv(PROVISIONAL_FILE, index=False)
    print(f"Saved and locked: {PROVISIONAL_FILE}")

Saved and locked: /content/drive/MyDrive/GenAI-task-affordance-labor-outcomes-dev/02_Task_Data_and_Classification/Output/Occupational_Task_Classification_Provisional_v6.csv
